# AIMET AdaScale Quantization for Gemma3-4B Language Model

This notebook shows a working code example of how to use AIMET AdaScale to optimize Gemma3-4B-IT Language Model weights for quantization.

The output of this notebook (saved model weights) can be used as `optimized_model_id` in `gemma3_4b.ipynb` to replace the SpinQuant weights.

---
### Required packages
The notebook assumes AIMET and Gemma3 related packages are already installed.

In [ ]:
# Install packages only if running in jupyter notebook mode
if hasattr(__builtins__,'__IPYTHON__'):
    !sudo -H apt-get -qq update
    !sudo -H apt-get -qq install libc++-dev
    !sudo -H pip install --quiet --upgrade --root-user-action=ignore --no-cache-dir transformers==4.50.0
    !sudo -H pip install --quiet --upgrade --root-user-action=ignore --no-cache-dir tokenizers==0.21.4
    !sudo -H pip install --quiet --upgrade --root-user-action=ignore --no-cache-dir datasets

### Overall flow
This notebook covers the following:
1. Parametrizing the Environment
2. Instantiate and evaluate HuggingFace Gemma3ForCausalLM model
3. Quantize model using AIMET AdaScale
4. Evaluate and Export AdaScale model

### What this notebook is not
* This notebook is not intended to show the full scope of optimization. For example, the flow will not use QAT, KD-QAT as deliberate choice to have the notebook execute more quickly.

### How to use the output
After running this notebook, set `optimized_model_id` in `gemma3_4b.ipynb` to the value of `adascale_dir` defined below.

### 1.1 Notebook Configs

In [ ]:
import os

# Feature knobs
context_length = int(os.getenv("CONTEXT_LENGTH", 2048))

# Quantization knobs
adascale_iterations = int(os.getenv("ADASCALE_ITERATIONS", 2000))

# Speed knobs
run_ppl_eval = os.getenv("RUN_PPL_EVAL", 'True').lower() in ('true', '1', 't')
# enable_bf16 = os.getenv("ENABLE_BF16", 'False').lower() in ('true', '1', 't')
num_eval_batches = int(os.getenv("NUM_EVAL_BATCHES", 0))

# Whether to load SpinQuant weights as the starting point for AdaScale
# Set to True to chain SpinQuant -> AdaScale -> SeqMSE pipeline
load_spinquant_weights = os.getenv("LOAD_SPINQUANT_WEIGHTS", 'True').lower() in ('true', '1', 't')

### 1.2 Setting NSP Target

In [ ]:
# Select quantsim config based on target
htp_config_file = "htp_quantsim_config_v81.json"

---
### 2. Instantiate and evaluate HuggingFace model

In [ ]:
import torch
from aimet_torch.utils import place_model, change_tensor_device_placement
from genai_lib.common.debug.profiler import event_marker

model_name = os.getenv("MODEL_NAME", 'gemma_4b_adascale')
model_id = "google/gemma-3-4b-it"  # HF checkpoint
cache_dir = os.getenv("CACHE_DIR", '/tmp/cache_dir')

# Output directory for the AdaScale-optimized model weights.
# After running this notebook, set optimized_model_id = adascale_dir in gemma3_4b.ipynb
adascale_dir = os.getenv("ADASCALE_DIR", "/tmp/output_dir/adascale")
os.makedirs(adascale_dir, exist_ok=True)

# If load_spinquant_weights=True, load SpinQuant weights as the starting point
spinquant_dir = os.getenv("SPINQUANT_DIR", "/tmp/output_dir/spinquant")

In [ ]:
# Recipe_logger: Initialize the logger and log environment details
from genai_lib.common.debug.recipe_logger import llm_lib_log_env_info, recipe_dump_init

recipe_dump_init(adascale_dir, "genai_lib_debug")
llm_lib_log_env_info()

#### 2.1 Load HF Model

We load `Gemma3ForCausalLM` (the language model only, not the full multimodal `Gemma3ForConditionalGeneration`).
Gemma3 already has separate `q_proj`, `k_proj`, `v_proj`, `gate_proj`, `up_proj` projections — no unpacking is needed.

In [ ]:
from transformers import AutoConfig, AutoTokenizer
from transformers.models.gemma3 import modeling_gemma3

llm_config = AutoConfig.from_pretrained(model_id, cache_dir=cache_dir, trust_remote_code=True)
# Use the text_config for the language model
text_config = llm_config.text_config

num_hidden_layers = int(os.getenv("NUM_HIDDEN_LAYERS", 0))
text_config.num_hidden_layers = num_hidden_layers if num_hidden_layers > 0 else text_config.num_hidden_layers

print(f'num_layer: {text_config.num_hidden_layers}, context_length: {context_length}, '
      f'num_attention_heads: {text_config.num_attention_heads}, num_kv_heads: {text_config.num_key_value_heads}')

# Determine the source of model weights
language_model_source = spinquant_dir if load_spinquant_weights else model_id
print(f'Loading language model weights from: {language_model_source}')

with event_marker('HuggingFace FP model creation'):
    # Load Gemma3ForCausalLM (language model only)
    model = modeling_gemma3.Gemma3ForCausalLM.from_pretrained(
        language_model_source,
        config=text_config,
        cache_dir=cache_dir,
        # torch_dtype=torch.bfloat16 if enable_bf16 else torch.float32
    )
    model = model.eval()
    os.environ['TOKENIZERS_PARALLELISM'] = '0'
    tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir, use_fast=True, trust_remote_code=True)
    # Adjust the tokenizer to limit to context_length
    tokenizer.model_max_length = context_length

#### 2.2 Instantiate Dataloaders

Load dataloaders for Wikitext and AdaScale-specific dataloaders.
Wikitext will be used for PPL evaluation, while C4 will be used for AdaScale.

In [ ]:
from llm_utils.wikitext_dataloader import get_wiki_dataset
from llm_utils.generic_dataloader import get_local_dataset

with event_marker("Instantiate dataloaders"):
    _, wikitext_test_dataloader, _ = get_wiki_dataset(context_length, tokenizer, cache_dir)

with event_marker("Instantiate adascale dataloaders"):
    adascale_train_dataloader, _ = get_local_dataset(
        context_length,
        tokenizer,
        json_path=os.getenv("C4_DATASET_PATH", "c4-train.00000-of-01024.json"),
        key="input_ids",
        batch_size=int(os.getenv("BATCH_SIZE", 2)),
        percent_dataset_to_load=int(os.getenv("PERCENT_DATASET_TO_LOAD", 1)),
        num_samples=int(os.getenv("NUM_SAMPLES", 500))
    )

#### 2.3 Eval HF Model

In [ ]:
from genai_lib.llm.evaluation_utils import llm_evaluate_ppl_with_dataloader

if run_ppl_eval:
    with event_marker("HuggingFace FP model eval"):
        with place_model(model, torch.device('cuda')):
            orig_ppl = llm_evaluate_ppl_with_dataloader(
                model=model, dataloader=wikitext_test_dataloader, num_batches=num_eval_batches
            )
    print(f"PPL score of HuggingFace FP model = {orig_ppl}")

In [ ]:
from genai_lib.common.debug.recipe_logger import llm_lib_log_property, Property
from genai_lib.common.debug.recipe_logger import llm_lib_log_metric, ModelType, Metric

# Recipe_logger: Log the context_length property and the metrics.
llm_lib_log_property({Property.context_length: context_length})

if run_ppl_eval:
    llm_lib_log_metric(ModelType.hf_model, Metric.ppl, orig_ppl)

---
## 3. AdaScale

The _AdaScale_ step is the primary focus of this notebook. This section modifies the model's weights to perform better with quantization.

#### 3.1 Configure AdaScale model config for Gemma3

Register the Gemma3 model structure with the AdaScale optimizer so it knows which blocks to optimize.

In [ ]:
from aimet_torch.experimental.adascale import adascale_optimizer

# Register Gemma3TextModel as the base model and Gemma3DecoderLayer as the block type for AdaScale
adascale_optimizer.adascale_model_config_dict[modeling_gemma3.Gemma3TextModel] = \
    adascale_optimizer.AdaScaleModelConfig(
        block_type=modeling_gemma3.Gemma3DecoderLayer,
        beta_gamma_lr=1e-3,
        scales_lr=5e-4
    )

#### 3.2 Redefine forward for JIT tracing in QuantSim Creation

AIMET requires KV Cache to be of type Tuple during the forward pass, so we wrap the forward to convert the KV Cache during inference.

In [ ]:
import types
from transformers import DynamicCache

def custom_forward(self, input_ids=None, attention_mask=None, position_ids=None, past_key_values=None, *args, **kwargs):
    past_key_values = DynamicCache.from_legacy_cache(past_key_values)

    lm_logits, new_past_key_values = self.__original_forward__(
        input_ids=input_ids,
        attention_mask=attention_mask,
        position_ids=position_ids,
        past_key_values=past_key_values,
        num_logits_to_return=0,
        return_dict=False,
        *args,
        **kwargs,
    )

    return lm_logits, new_past_key_values.to_legacy_cache()

# Save original forward method and replace with custom forward
model.__original_forward__ = model.forward
model.forward = types.MethodType(custom_forward, model)

#### 3.3 Create QuantSim configured for QNN HTP target

In [ ]:
from aimet_common.defs import QuantScheme
from aimet_torch.v2.quantsim import QuantizationSimModel

dummy_input = torch.randint(0, 1, (1, 1, context_length), device="cuda")

with event_marker("create KVCache Quantsim"):
    with place_model(model, "cuda"):
        model.config.return_dict = False
        quantsim = QuantizationSimModel(
            model=model,
            quant_scheme=QuantScheme.post_training_tf,
            default_output_bw=16,
            default_param_bw=4,
            in_place=True,
            dummy_input=tuple(list(dummy_input)),
            config_file=htp_config_file
        )

In [ ]:
from aimet_torch.v2.experimental import propagate_output_encodings
from aimet_torch.nn.modules import custom as aimet_ops

propagate_output_encodings(quantsim, aimet_ops.Concat)

#### 3.4 Enable per-channel quantization

Configure linear quantizers to support per-channel quantization to mimic Convolution layer behavior on-device.

In [ ]:
from aimet_torch.v2.nn.true_quant import QuantizedLinear
from aimet_torch.v2.quantization.affine import QuantizeDequantize

for name, qmodule in quantsim.named_qmodules():
    if isinstance(qmodule, QuantizedLinear):
        assert len(qmodule.weight.shape) == 2, (
            f"Per-channel quantization for linear weights is only supported for 2D weights, "
            f"got shape: {qmodule.weight.shape} instead"
        )
        qmodule.param_quantizers["weight"] = QuantizeDequantize(
            shape=(qmodule.weight.shape[0], 1),
            bitwidth=qmodule.param_quantizers["weight"].bitwidth,
            symmetric=qmodule.param_quantizers["weight"].symmetric
        ).to(next(quantsim.model.parameters()).device)

#### 3.5 Manual mixed precision + Disable un-needed quantizers

AdaScale only operates on the decoder blocks of the model, therefore we disable quantizers on non-decoder blocks.
We also increase bitwidth for RMSNorm due to higher quantization sensitivity.

In [ ]:
import re

# Remove quantizers for non-decoder blocks (embedding table and lm_head)
quantsim.model.model.embed_tokens.param_quantizers["weight"] = None
quantsim.model.lm_head.param_quantizers["weight"] = None

# Increase bitwidth for RMSNorm due to higher quantization sensitivity
for name, qmodule in quantsim.named_qmodules():
    if re.search(r'rmsnorm', qmodule.__class__.__name__.lower()):
        qmodule.param_quantizers['weight'] = QuantizeDequantize(
            shape=(), bitwidth=16, symmetric=False
        ).to(next(quantsim.model.parameters()).device)

#### 3.6 Apply AdaScale

Apply AdaScale to optimize weights for quantization.

In [ ]:
param_devices = set(p.device for p in quantsim.model.parameters())
print(f"Model parameter devices: {param_devices}")

# Check GPU memory usage
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"GPU memory reserved:  {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

In [ ]:
from aimet_torch.experimental.adascale import apply_adascale

quantsim.model.cuda()

with event_marker("apply AdaScale", flush_ram=True):
    with torch.no_grad():
        with place_model(quantsim.model, "cuda"):
            apply_adascale(
                qsim=quantsim,
                data_loader=adascale_train_dataloader,
                forward_fn=custom_forward,
                num_iterations=adascale_iterations
            )

---
## 4. Evaluate and Export Model

#### 4.1 AdaScale Evaluation

Evaluate the AdaScale-optimized model in FP mode (activation quantizers removed) to verify the weight transformation did not degrade accuracy.

In [ ]:
from aimet_torch.v2.utils import remove_activation_quantizers

if run_ppl_eval:
    with event_marker("AdaScale FP model eval"):
        with place_model(quantsim.model, torch.device('cuda')), remove_activation_quantizers(quantsim.model):
            adascaled_ppl = llm_evaluate_ppl_with_dataloader(
                model=quantsim.model, dataloader=wikitext_test_dataloader, num_batches=num_eval_batches
            )
    print(f"PPL score of AdaScale model = {adascaled_ppl}")

In [ ]:
if run_ppl_eval:
    llm_lib_log_metric(ModelType.hf_model, Metric.ppl, adascaled_ppl, model_name="adascale")

#### 4.2 Export model

This notebook exports new model weights that are more quantization friendly, and stores them in the specified `adascale_dir`.

**To use these weights in `gemma3_4b.ipynb`:**
Set `optimized_model_id = adascale_dir` (e.g., `"/tmp/output_dir/adascale"`) in `gemma3_4b.ipynb`.

In [ ]:
with event_marker("save AdaScale model", flush_ram=True):
    fp_ada_model = QuantizationSimModel.get_original_model(quantsim.model, qdq_weights=True)
    fp_ada_model.save_pretrained(adascale_dir)
    tokenizer.save_pretrained(adascale_dir)

print(f"AdaScale model saved to: {adascale_dir}")
print(f"In gemma3_4b.ipynb, set: optimized_model_id = \"{adascale_dir}\"")

---
### Summary

In [ ]:
from genai_lib.common.debug.profiler import EventProfiler
EventProfiler().report()
EventProfiler().json_dump(os.path.join(adascale_dir, 'profiling_stats.json'))

Copyright (c) Qualcomm Technologies, Inc. and/or its subsidiaries.   
All rights reserved.   
Confidential and Proprietary - Qualcomm Technologies, Inc.  